# Move Operator Comparison

This notebook compares `move_first`, `move_best`, and `move_plateau`.

It produces two LaTeX tables:

1. solution quality and winner rate
2. runtime, number of moves, and number of passes

The analysis is grouped by graph type, size class, and density regime.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [2]:
MOVE_OPERATORS = [
    "move_first",
    "move_best",
    "move_plateau",
]

GRAPH_ORDER = ["powerlaw", "er"]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

RAW_RESULTS_FILE = Path("../../results/experiment2/move_operator/raw_results.csv")

## Load data

In [3]:
raw = pd.read_csv(RAW_RESULTS_FILE)

raw["dataset_group"] = raw["size_class"].astype(str) + " " + raw["regime"].astype(str)

raw["graph_type"] = pd.Categorical(
    raw["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

raw["dataset_group"] = pd.Categorical(
    raw["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

raw["pipeline"] = pd.Categorical(
    raw["pipeline"],
    categories=MOVE_OPERATORS,
    ordered=True,
)

raw = raw[raw["pipeline"].isin(MOVE_OPERATORS)].copy()

# Solution quality

For every instance and operator, only the run with the highest final solution quality is retained.

The best result across all compared operators is used as the reference. Relative solution quality of an operator is defined as

$
\frac{\text{best solution quality on the instance}}
     {\text{solution quality of the operator}}.
$

A value of $1.0$ means that the operator matches the best result found on the same instance. Values greater than $1.0$ indicate the remaining quality gap.

In [4]:
best_run_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "pipeline",
]

best_runs = (
    raw
    .sort_values("final_density", ascending=False)
    .groupby(best_run_keys, observed=True,as_index=False,)
    .head(1)
    .reset_index(drop=True)
)

In [5]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = best_runs.pivot_table(
    index=instance_keys,
    columns="pipeline",
    values="final_density",
    observed=True,
)[MOVE_OPERATORS]

best_per_instance = density_table.max(axis=1)

relative_to_best = density_table.rdiv(best_per_instance, axis=0)

quality_summary = (
    relative_to_best
    .groupby(level=["graph_type", "dataset_group"])
    .mean()
    .stack()
    .rename("mean_relative_to_best")
    .reset_index()
    .rename(columns={"pipeline": "operator"})
)

quality_summary

,graph_type,dataset_group,operator,mean_relative_to_best
0,powerlaw,small sparse,move_first,1.016997
1,powerlaw,small sparse,move_best,1.017767
2,powerlaw,small sparse,move_plateau,1.000000
3,powerlaw,small dense,move_first,1.009506
4,powerlaw,small dense,move_best,1.010991
5,powerlaw,small dense,move_plateau,1.000012
6,powerlaw,large sparse,move_first,1.011670
7,powerlaw,large sparse,move_best,1.011699
8,powerlaw,large sparse,move_plateau,1.000000
9,powerlaw,large dense,move_first,1.003999


In [6]:
winner_summary = (
    density_table
    .eq(best_per_instance, axis=0)
    .groupby(level=["graph_type", "dataset_group"])
    .mean()
    .stack()
    .rename("winner_rate")
    .reset_index()
    .rename(columns={"pipeline": "operator"})
)

winner_summary

,graph_type,dataset_group,operator,winner_rate
0,powerlaw,small sparse,move_first,0.004
1,powerlaw,small sparse,move_best,0.004
2,powerlaw,small sparse,move_plateau,1.000
3,powerlaw,small dense,move_first,0.028
4,powerlaw,small dense,move_best,0.032
5,powerlaw,small dense,move_plateau,0.996
6,powerlaw,large sparse,move_first,0.000
7,powerlaw,large sparse,move_best,0.000
8,powerlaw,large sparse,move_plateau,1.000
9,powerlaw,large dense,move_first,0.000


## Runtime

The reported runtime is the total local-search runtime of all runs for one instance and operator, averaged over all instances in the corresponding dataset group.

In [7]:
runtime_per_instance = (
    raw
    .groupby(best_run_keys, observed=True)
    .agg(
        total_runtime=("ls_runtime", "sum"),
        moves_per_run=("num_moves", "mean"),
        passes_per_run=("num_passes", "mean"),
    )
    .reset_index()
)

runtime_summary = (
    runtime_per_instance
    .groupby(["graph_type", "dataset_group", "pipeline"], observed=True, as_index=False)
    .agg(
        mean_runtime=("total_runtime", "mean"),
        mean_moves_per_run=("moves_per_run", "mean"),
        mean_passes_per_run=("passes_per_run", "mean"),
    )
    .rename(columns={"pipeline": "operator"})
)

runtime_summary

,graph_type,dataset_group,operator,mean_runtime,mean_moves_per_run,mean_passes_per_run
0,powerlaw,small sparse,move_first,3.613431,32.91950,33.91950
1,powerlaw,small sparse,move_best,19.042124,30.16770,31.16770
2,powerlaw,small sparse,move_plateau,12.792214,349.54160,25.40590
3,powerlaw,small dense,move_first,5.554444,28.51580,29.51580
4,powerlaw,small dense,move_best,36.807819,26.11395,27.11395
5,powerlaw,small dense,move_plateau,18.937005,351.66415,16.60415
6,powerlaw,large sparse,move_first,41.693901,211.11080,212.11080
7,powerlaw,large sparse,move_best,996.811346,197.00375,198.00375
8,powerlaw,large sparse,move_plateau,103.475413,2328.52955,21.29505
9,powerlaw,large dense,move_first,303.365740,135.14725,136.14725


## Combined comparison data

In [8]:
comparison_summary = (
    quality_summary
    .merge(winner_summary, on=["graph_type", "dataset_group", "operator"])
    .merge(runtime_summary, on=["graph_type", "dataset_group", "operator"])
)

comparison_summary["operator"] = pd.Categorical(
    comparison_summary["operator"],
    categories=MOVE_OPERATORS,
    ordered=True,
)

comparison_summary = (
    comparison_summary
    .sort_values(["graph_type", "dataset_group", "operator"])
    .reset_index(drop=True)
)

comparison_summary

,graph_type,dataset_group,operator,mean_relative_to_best,winner_rate,mean_runtime,mean_moves_per_run,mean_passes_per_run
0,powerlaw,small sparse,move_first,1.016997,0.004,3.613431,32.91950,33.91950
1,powerlaw,small sparse,move_best,1.017767,0.004,19.042124,30.16770,31.16770
2,powerlaw,small sparse,move_plateau,1.000000,1.000,12.792214,349.54160,25.40590
3,powerlaw,small dense,move_first,1.009506,0.028,5.554444,28.51580,29.51580
4,powerlaw,small dense,move_best,1.010991,0.032,36.807819,26.11395,27.11395
5,powerlaw,small dense,move_plateau,1.000012,0.996,18.937005,351.66415,16.60415
6,powerlaw,large sparse,move_first,1.011670,0.000,41.693901,211.11080,212.11080
7,powerlaw,large sparse,move_best,1.011699,0.000,996.811346,197.00375,198.00375
8,powerlaw,large sparse,move_plateau,1.000000,1.000,103.475413,2328.52955,21.29505
9,powerlaw,large dense,move_first,1.003999,0.000,303.365740,135.14725,136.14725


## LaTeX helper functions

In [9]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_operator(operator: str) -> str:
    return r"\texttt{" + operator.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{value:.{decimals}f}"


def format_percent(value: float, decimals: int = 1) -> str:
    return rf"{truncate_number(100 * value, decimals):.{decimals}f}\,\%"

## Build LaTeX tables

In [10]:
def make_quality_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": r"Erdős-Rényi",
    }

    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lllrr}",
        r"\toprule",
        r"Graph type & Dataset & Operator & \shortstack{Mean relative\\solution quality} & Win rate \\",
        r"\midrule",
    ]

    for graph_type in GRAPH_ORDER:
        graph_df = df[df["graph_type"] == graph_type]

        for dataset_index, dataset in enumerate(DATASET_ORDER):
            part = graph_df[graph_df["dataset_group"] == dataset].sort_values("operator")

            for i, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{12}}{{*}}{{{graph_labels[graph_type]}}}"
                    if dataset_index == 0 and i == 0
                    else ""
                )
                dataset_cell = (
                    rf"\multirow{{3}}{{*}}{{{dataset}}}"
                    if i == 0
                    else ""
                )

                quality = format_number(row.mean_relative_to_best, 6)
                winner_rate = format_percent(row.winner_rate, 1)

                if row.operator == "move_plateau":
                    quality = rf"\textbf{{{quality}}}"
                    winner_rate = rf"\textbf{{{winner_rate}}}"

                lines.append(
                    f"{graph_cell} & {dataset_cell} & {latex_operator(str(row.operator))} & {quality} & {winner_rate} \\\\"
                )

            if dataset_index < len(DATASET_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-5}")
            else:
                lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.extend(
        [
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [11]:
quality_latex = make_quality_latex_table(
    comparison_summary,
    caption=(
        "Mean relative solution quality and win rate of the move operators on Powerlaw and Erdős-Rényi instances."
    ),
    label="tab:move_quality",
)

print(quality_latex)

\begin{table}[!htbp]
\centering
\caption{Mean relative solution quality and win rate of the move operators on Powerlaw and Erdős-Rényi instances.}
\label{tab:move_quality}
\begin{tabular}{lllrr}
\toprule
Graph type & Dataset & Operator & \shortstack{Mean relative\\solution quality} & Win rate \\
\midrule
\multirow{12}{*}{Powerlaw} & \multirow{3}{*}{small sparse} & \texttt{move\_first} & 1.016997 & 0.4\,\% \\
 &  & \texttt{move\_best} & 1.017767 & 0.4\,\% \\
 &  & \texttt{move\_plateau} & \textbf{1.000000} & \textbf{100.0\,\%} \\
\cmidrule(l){2-5}
 & \multirow{3}{*}{small dense} & \texttt{move\_first} & 1.009506 & 2.8\,\% \\
 &  & \texttt{move\_best} & 1.010991 & 3.2\,\% \\
 &  & \texttt{move\_plateau} & \textbf{1.000012} & \textbf{99.6\,\%} \\
\cmidrule(l){2-5}
 & \multirow{3}{*}{large sparse} & \texttt{move\_first} & 1.011670 & 0.0\,\% \\
 &  & \texttt{move\_best} & 1.011699 & 0.0\,\% \\
 &  & \texttt{move\_plateau} & \textbf{1.000000} & \textbf{100.0\,\%} \\
\cmidrule(l){2-5}
 & \mul

In [12]:
def make_runtime_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": r"Erdős-Rényi",
    }

    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lllrrr}",
        r"\toprule",
        r"Graph type & Dataset & Operator & Run time (s) & Moves & Rounds \\",
        r"\midrule",
    ]

    for graph_type in GRAPH_ORDER:
        graph_df = df[df["graph_type"] == graph_type]

        for dataset_index, dataset in enumerate(DATASET_ORDER):
            part = graph_df[graph_df["dataset_group"] == dataset].sort_values("operator")

            for i, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{12}}{{*}}{{{graph_labels[graph_type]}}}"
                    if dataset_index == 0 and i == 0
                    else ""
                )
                dataset_cell = (
                    rf"\multirow{{3}}{{*}}{{{dataset}}}"
                    if i == 0
                    else ""
                )

                lines.append(
                    f"{graph_cell} & {dataset_cell} "
                    f"& {latex_operator(str(row.operator))} "
                    f"& {format_number(row.mean_runtime, 2)} "
                    f"& {format_number(row.mean_moves_per_run, 1)} "
                    f"& {format_number(row.mean_passes_per_run, 1)} \\\\"
                )

            if dataset_index < len(DATASET_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-6}")
            else:
                lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.extend(
        [
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [13]:
runtime_latex = make_runtime_latex_table(
    comparison_summary,
    caption=(
        "Mean total run time of the ten runs and average number of vertex moves and rounds on Powerlaw and Erdős-Rényi instances."
    ),
    label="tab:move_runtime",
)

print(runtime_latex)

\begin{table}[!htbp]
\centering
\caption{Mean total run time of the ten runs and average number of vertex moves and rounds on Powerlaw and Erdős-Rényi instances.}
\label{tab:move_runtime}
\begin{tabular}{lllrrr}
\toprule
Graph type & Dataset & Operator & Run time (s) & Moves & Rounds \\
\midrule
\multirow{12}{*}{Powerlaw} & \multirow{3}{*}{small sparse} & \texttt{move\_first} & 3.61 & 32.9 & 33.9 \\
 &  & \texttt{move\_best} & 19.04 & 30.2 & 31.2 \\
 &  & \texttt{move\_plateau} & 12.79 & 349.5 & 25.4 \\
\cmidrule(l){2-6}
 & \multirow{3}{*}{small dense} & \texttt{move\_first} & 5.55 & 28.5 & 29.5 \\
 &  & \texttt{move\_best} & 36.81 & 26.1 & 27.1 \\
 &  & \texttt{move\_plateau} & 18.94 & 351.7 & 16.6 \\
\cmidrule(l){2-6}
 & \multirow{3}{*}{large sparse} & \texttt{move\_first} & 41.69 & 211.1 & 212.1 \\
 &  & \texttt{move\_best} & 996.81 & 197.0 & 198.0 \\
 &  & \texttt{move\_plateau} & 103.48 & 2328.5 & 21.3 \\
\cmidrule(l){2-6}
 & \multirow{3}{*}{large dense} & \texttt{move\_first} & 3